# 04 Evaluation And Prediction

This notebook shows how to score rows with the trained pipeline object. It trains a small temporary model for demonstration so the notebook does not depend on committed binary artifacts.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from fault_prediction.config import DEFAULT_DATA_PATH, RANDOM_STATE, TARGET_COLUMN
from fault_prediction.data import load_factory_data
from fault_prediction.models import predict_scores, train_model

df = load_factory_data(DEFAULT_DATA_PATH)
train_df, holdout_df = train_test_split(
    df,
    train_size=1000,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)
result = train_model(train_df, profile="fast", n_jobs=1)
artifact = result.artifact

In [ ]:
model = artifact["model"]
threshold = artifact["threshold"]
X_holdout = holdout_df.drop(columns=[TARGET_COLUMN]).head(20)
probabilities = predict_scores(model, X_holdout)
predictions = (probabilities >= threshold).astype(int)

prediction_preview = X_holdout[["Unique ID", "Product ID"]].copy()
prediction_preview["predicted_probability_fault"] = probabilities
prediction_preview["predicted_machine_status"] = predictions
prediction_preview

In [ ]:
pd.Series(result.metrics, name="validation_metric")

## Production Prediction

After training a saved artifact with the CLI, score a dataset with:

```bash
fault-predict predict --model models/fault_voting_classifier.joblib --input data/raw/factory_data.csv
```